# Indonesian Labor Law RAG — Parsing Walkthrough

Notebook ini dibuat supaya proses parsing bisa dilihat **step-by-step** sampai menjadi dataset `.jsonl` yang siap masuk ke tahap embedding/indexing.

Pipeline:

```text
PDF
→ inspect native/scanned page
→ PyMuPDF word extraction
→ reconstruct visual lines
→ normalization & noise cleanup
→ detect BAB / Bagian / Paragraf / Pasal / Ayat
→ structure-aware chunking
→ build embedding_text
→ validation
→ save JSONL
```

### Chunking policy

Prioritas struktur mengikuti PRD:

```text
BAB → Bagian → Pasal → Ayat
```

Implementasi:
- Jika suatu **Pasal punya Ayat**, maka **1 Ayat = 1 chunk**.
- Jika Pasal **tidak punya Ayat**, maka **1 Pasal = 1 chunk**.
- `BAB`, `Bagian`, dan `Paragraf` disimpan sebagai metadata/context.
- Unit yang sangat panjang baru di-split sebagai fallback.
- `PENJELASAN` dipisahkan dari `batang_tubuh`.

## 0. Dependencies

Parser utama memakai **PyMuPDF** (`fitz`).

OCR hanya dipakai sebagai fallback jika suatu halaman tidak memiliki text layer yang cukup.

In [1]:
# Jalankan jika dependency belum tersedia.
# %pip install pymupdf pandas pillow pytesseract

import fitz  # PyMuPDF
import pandas as pd

print("PyMuPDF version:", fitz.VersionBind)

PyMuPDF version: 1.26.7


## 1. Konfigurasi source PDF

In [2]:
from __future__ import annotations

import hashlib
import json
import re
import statistics
import unicodedata
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable, Optional

DATA_DIR = Path("/mnt/data")

UU13_PATH = DATA_DIR / "1 UU No. 13 Tahun 2003 tentang Ketenagakerjaan(1).pdf"
PP35_PATH = DATA_DIR / "2 PP No. 35 Tahun 2021 tentang PKWT, Alih Daya, Waktu Kerja, dan PHK(1).pdf"

OUTPUT_PATH = DATA_DIR / "labor_law_chunks_notebook.jsonl"

@dataclass(frozen=True)
class SourceDocument:
    document_id: str
    document_name: str
    path: Path

SOURCES = [
    SourceDocument(
        document_id="uu13_2003",
        document_name="UU No. 13 Tahun 2003 tentang Ketenagakerjaan",
        path=UU13_PATH,
    ),
    SourceDocument(
        document_id="pp35_2021",
        document_name="PP No. 35 Tahun 2021 tentang PKWT, Alih Daya, Waktu Kerja, dan PHK",
        path=PP35_PATH,
    ),
]

for source in SOURCES:
    print(source.document_id, "->", source.path)
    assert source.path.exists(), f"File tidak ditemukan: {source.path}"

uu13_2003 -> /mnt/data/1 UU No. 13 Tahun 2003 tentang Ketenagakerjaan(1).pdf
pp35_2021 -> /mnt/data/2 PP No. 35 Tahun 2021 tentang PKWT, Alih Daya, Waktu Kerja, dan PHK(1).pdf


## 2. Inspect PDF

Di sini kita cek jumlah halaman dan seberapa banyak text layer native yang tersedia.

Kalau text layer terlalu sedikit, halaman bisa diarahkan ke OCR fallback.

In [3]:
def inspect_pdf(pdf_path: Path, sample_pages: int = 8) -> pd.DataFrame:
    doc = fitz.open(pdf_path)
    rows = []

    for page_idx in range(min(sample_pages, len(doc))):
        page = doc[page_idx]
        words = page.get_text("words")
        text = " ".join(w[4] for w in words)
        rows.append(
            {
                "page": page_idx + 1,
                "word_count": len(words),
                "native_alnum": len(re.sub(r"\W", "", text)),
                "likely_scanned": len(re.sub(r"\W", "", text)) < 20,
            }
        )

    return pd.DataFrame(rows)

for source in SOURCES:
    print("\n", source.document_id)
    display(inspect_pdf(source.path))


 uu13_2003


,page,word_count,native_alnum,likely_scanned
0,1,184,1262,False
1,2,186,1046,False
2,3,267,1718,False
3,4,258,1662,False
4,5,275,1784,False
5,6,161,1067,False
6,7,141,871,False
7,8,178,1141,False



 pp35_2021


,page,word_count,native_alnum,likely_scanned
0,1,164,978,False
1,2,151,910,False
2,3,229,1349,False
3,4,210,1378,False
4,5,123,825,False
5,6,158,887,False
6,7,189,1060,False
7,8,183,1093,False


## 3. Kenapa pakai `page.get_text("words")`?

Bukan langsung `page.get_text("text")`.

Pada PDF regulasi, marker seperti `(1)`, `(2)`, atau nomor Pasal kadang berada di PDF text block yang berbeda dari kalimatnya. Dengan mode `words`, kita dapat koordinat `(x0, y0, x1, y1)` setiap kata dan bisa menyusun ulang **visual line** berdasarkan posisi.

In [4]:
def show_raw_words(pdf_path: Path, page_number: int = 7, limit: int = 40) -> pd.DataFrame:
    doc = fitz.open(pdf_path)
    page = doc[page_number - 1]
    words = page.get_text("words")

    rows = [
        {
            "x0": round(w[0], 1),
            "y0": round(w[1], 1),
            "x1": round(w[2], 1),
            "y1": round(w[3], 1),
            "word": w[4],
            "block": w[5],
            "line": w[6],
        }
        for w in words[:limit]
    ]
    return pd.DataFrame(rows)

display(show_raw_words(PP35_PATH, page_number=7))

,x0,y0,x1,y1,word,block,line
0,277.9,163.1,336.7,178.9,PRESIDEN,0,0
1,241.9,175.0,301.3,191.5,REPUBLIK,0,1
2,307.4,177.1,371.6,192.2,TNDONESIA,0,1
3,293.8,212.9,308.4,235.6,-7,1,0
4,313.7,218.8,317.5,234.6,-,1,0
5,284.9,245.8,316.1,262.9,Pasal,2,0
6,319.9,246.5,326.6,263.0,6,2,0
7,191.5,265.9,247.8,283.1,Pekerjaan,3,0
8,252.7,265.1,280.9,283.0,yang,3,0
9,284.9,265.1,358.4,283.6,diperkirakan,3,0


## 4. Reconstruct visual lines dari koordinat kata

In [5]:
def _native_word_lines(page: fitz.Page) -> list[str]:
    words = page.get_text("words")
    if not words:
        return []

    heights = [max(1.0, w[3] - w[1]) for w in words]
    median_height = statistics.median(heights)
    y_tolerance = max(3.0, min(8.0, median_height * 0.40))

    ordered = sorted(words, key=lambda w: (((w[1] + w[3]) / 2.0), w[0]))
    grouped = []

    for word in ordered:
        y_center = (word[1] + word[3]) / 2.0

        if grouped and abs(y_center - grouped[-1][0]) <= y_tolerance:
            grouped[-1][1].append(word)
            grouped[-1][0] = (
                sum((w[1] + w[3]) / 2.0 for w in grouped[-1][1])
                / len(grouped[-1][1])
            )
        else:
            grouped.append([y_center, [word]])

    lines = []
    for _, line_words in grouped:
        line_words = sorted(line_words, key=lambda w: w[0])
        lines.append(" ".join(w[4] for w in line_words))

    return lines


doc = fitz.open(PP35_PATH)
sample_page = doc[6]  # physical PDF page 7
raw_lines = _native_word_lines(sample_page)

for i, line in enumerate(raw_lines[:35], start=1):
    print(f"{i:02d}: {line}")

01: PRESIDEN
02: REPUBLIK TNDONESIA
03: -7 -
04: Pasal 6
05: Pekerjaan yang diperkirakan penyelesaiannya dalam waktu
06: yang tidak terlalu lama sebagaimana dimaksud dalam
07: Pasal 5 ayat (1) huruf a dilaksanakan paling lama 5 (lima)
08: tahun.
09: Pasal 7
10: (1) Pekerjaan yang bersifat musiman sebagaimana
11: dimaksud dalam Pasal 5 ayat (1) huruf b merupakan
12: pekerj aan yang pelaksanaannya tergantung pada:
13: a. musim atau cuaca; atau
14: b. kondisi tertentu.
15: (21 Pekerjaan yang pelaksanaannya tergantung pada
16: musim atau cuaca sebagaimana dimaksud pada
17: ayat (1) huruf a hanya dapat dilakukan pada musim
18: tertentu atau cuaca tertentu.
19: (3) Pekerjaan yang pelaksanaannya tergantung pada
20: kondisi tertentu sebagaimana dimaksud pada ayat (1)
21: huruf b merupakan pekerjaan tambahan yang
22: dilakukan untuk memenuhi pesanan atau target
23: tertentu.
24: Pasal 8
25: (1) PKWT berdasarkan jangka waktu sebagaimana
26: dimaksud dalam Pasal 5 ayat (1) dapat dibuat untuk
27: 

## 5. Text normalization + conservative repair

Kita **tidak melakukan spell correction terhadap isi hukum**.

Repair hanya untuk marker struktur yang rusak akibat text layer/OCR, misalnya:
- `Pasal 1 1` → `Pasal 11`
- `(21 ...` → `(2) ...`
- `l0.` → `10.`

Tujuannya hanya agar parser bisa mengenali struktur.

In [6]:
HEADER_PATTERNS = [
    re.compile(r"^SALINAN$", re.I),
    re.compile(r"^PRES\s*IDEN$", re.I),
    re.compile(r"^REP\w*LIK\s+\w*NDONESIA$", re.I),
    re.compile(r"^-\s*\d+\s*-$"),
]

SK_RE = re.compile(r"^SK\s+No\b", re.I)
BAB_RE = re.compile(r"^BAB\s+([IVXLCDM]+)$", re.I)
BAGIAN_RE = re.compile(r"^Bagian\s+([A-Za-z]+)$", re.I)
PARAGRAF_RE = re.compile(r"^Paragraf\s+(\d+|[A-Za-z]+)$", re.I)
PASAL_RE = re.compile(r"^Pasal\s+(\d+[A-Za-z]?)$", re.I)
AYAT_RE = re.compile(r"^\((\d{1,2})\)\s*(.*)$")
EXPL_AYAT_RE = re.compile(r"^Ayat\s+\((\d{1,2})\)\s*(.*)$", re.I)
EXPL_SECTION_RE = re.compile(r"^(I|II)\.\s+(UMUM|PASAL\s+DEMI\s+PASAL)$", re.I)


def normalize_spaces(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u00ad", "")
    return re.sub(r"[ \t]+", " ", text).strip()


def repair_structure_marker(line: str) -> str:
    line = normalize_spaces(line)

    m = re.match(r"^(Pasal)\s+((?:\d\s*){1,4})([A-Za-z]?)\s*$", line, re.I)
    if m:
        number = re.sub(r"\s+", "", m.group(2))
        line = f"Pasal {number}{m.group(3)}"

    if line.startswith("(") and ")" not in line[:6]:
        m = re.match(r"^\((\d{1,2})[1Il](.*)$", line)
        if m and m.group(2) and not m.group(2).startswith(")"):
            line = f"({m.group(1)}) {m.group(2).lstrip()}"

    line = re.sub(r"^[lI](\d)\.", r"1\1.", line)
    line = re.sub(
        r"^(\d)\s+(\d)\.\s*",
        lambda m: f"{m.group(1)}{m.group(2)}. ",
        line,
    )

    return line


examples = [
    "Pasal 1 1",
    "(21 Perjanjian Kerja harian...",
    "l0.Perjanjian...",
    "  BAB   II  ",
]

for x in examples:
    print(repr(x), "=>", repr(repair_structure_marker(x)))

'Pasal 1 1' => 'Pasal 11'
'(21 Perjanjian Kerja harian...' => '(2) Perjanjian Kerja harian...'
'l0.Perjanjian...' => '10.Perjanjian...'
'  BAB   II  ' => 'BAB II'


## 6. Header/footer cleanup + OCR fallback

In [7]:
def is_page_header(line: str, line_index: int) -> bool:
    return line_index < 7 and any(pattern.match(line) for pattern in HEADER_PATTERNS)


def is_footer_teaser(line: str) -> bool:
    return bool(re.search(r"(?:\.\s*){2,}$", line)) or line.lower() == "bagian"


def canonical_structure(line: str) -> Optional[str]:
    line = repair_structure_marker(line)

    if (
        PASAL_RE.match(line)
        or BAB_RE.match(line)
        or BAGIAN_RE.match(line)
        or PARAGRAF_RE.match(line)
    ):
        return line.lower()

    m = AYAT_RE.match(line)
    if m and not m.group(2):
        return f"ayat-{m.group(1)}"

    return None


def _ocr_page(page: fitz.Page, lang: str = "ind+eng") -> list[str]:
    try:
        import pytesseract
        from PIL import Image
    except ImportError as exc:
        raise RuntimeError(
            "Scanned page detected, tapi dependency OCR belum tersedia. "
            "Install pytesseract + Pillow dan system package tesseract-ocr."
        ) from exc

    pix = page.get_pixmap(matrix=fitz.Matrix(2.0, 2.0), alpha=False)
    image = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    text = pytesseract.image_to_string(image, lang=lang, config="--psm 6")
    return text.splitlines()


def extract_pdf_pages(
    pdf_path: Path,
    *,
    enable_ocr: bool = True,
    ocr_lang: str = "ind+eng",
    min_native_alnum: int = 20,
) -> list[list[str]]:
    doc = fitz.open(pdf_path)
    pages = []

    for page in doc:
        native_lines = _native_word_lines(page)
        native_alnum = len(re.sub(r"\W", "", " ".join(native_lines)))

        if enable_ocr and native_alnum < min_native_alnum:
            raw_lines = _ocr_page(page, lang=ocr_lang)
        else:
            raw_lines = native_lines

        cleaned = []

        for idx, raw_line in enumerate(raw_lines):
            line = repair_structure_marker(raw_line)

            if not line:
                continue
            if is_page_header(line, idx):
                continue
            if SK_RE.match(line):
                continue

            cleaned.append(line)

        while cleaned and is_footer_teaser(cleaned[-1]):
            cleaned.pop()

        pages.append(cleaned)

    # Remove duplicate heading teaser lintas halaman.
    for idx in range(len(pages) - 1):
        if not pages[idx]:
            continue

        last = canonical_structure(pages[idx][-1])
        if not last:
            continue

        next_candidates = [canonical_structure(x) for x in pages[idx + 1][:10]]

        if last in next_candidates:
            pages[idx].pop()

    return pages

### Preview hasil extraction yang sudah dibersihkan

In [8]:
pp35_pages = extract_pdf_pages(PP35_PATH, enable_ocr=True)

print("Total physical pages:", len(pp35_pages))
print("\nPreview physical PDF page 7:\n")

for i, line in enumerate(pp35_pages[6][:35], start=1):
    print(f"{i:02d}: {line}")

Total physical pages: 56

Preview physical PDF page 7:

01: Pasal 6
02: Pekerjaan yang diperkirakan penyelesaiannya dalam waktu
03: yang tidak terlalu lama sebagaimana dimaksud dalam
04: Pasal 5 ayat (1) huruf a dilaksanakan paling lama 5 (lima)
05: tahun.
06: Pasal 7
07: (1) Pekerjaan yang bersifat musiman sebagaimana
08: dimaksud dalam Pasal 5 ayat (1) huruf b merupakan
09: pekerj aan yang pelaksanaannya tergantung pada:
10: a. musim atau cuaca; atau
11: b. kondisi tertentu.
12: (2) Pekerjaan yang pelaksanaannya tergantung pada
13: musim atau cuaca sebagaimana dimaksud pada
14: ayat (1) huruf a hanya dapat dilakukan pada musim
15: tertentu atau cuaca tertentu.
16: (3) Pekerjaan yang pelaksanaannya tergantung pada
17: kondisi tertentu sebagaimana dimaksud pada ayat (1)
18: huruf b merupakan pekerjaan tambahan yang
19: dilakukan untuk memenuhi pesanan atau target
20: tertentu.
21: Pasal 8
22: (1) PKWT berdasarkan jangka waktu sebagaimana
23: dimaksud dalam Pasal 5 ayat (1) dapat dibuat

## 7. Detect struktur hukum

Sekarang kita cek marker yang berhasil dikenali dari hasil extraction.

In [9]:
def classify_structure(line: str):
    if BAB_RE.match(line):
        return "BAB"
    if BAGIAN_RE.match(line):
        return "BAGIAN"
    if PARAGRAF_RE.match(line):
        return "PARAGRAF"
    if PASAL_RE.match(line):
        return "PASAL"
    if AYAT_RE.match(line):
        return "AYAT"
    if EXPL_AYAT_RE.match(line):
        return "EXPL_AYAT"
    if line.upper() == "PENJELASAN":
        return "PENJELASAN"
    return None


rows = []
for page_no, lines in enumerate(pp35_pages, start=1):
    for line in lines:
        kind = classify_structure(line)
        if kind:
            rows.append({"page": page_no, "kind": kind, "text": line})

structure_df = pd.DataFrame(rows)
display(structure_df.head(60))

,page,kind,text
0,2,BAB,BAB I
1,2,PASAL,Pasal 1
2,5,BAB,BAB II
3,5,BAGIAN,Bagian Kesatu
4,5,PASAL,Pasal 2
5,5,AYAT,(1) Hubungan Kerja terjadi karena adanya Perja...
6,5,AYAT,(2) Perjanjian Kerja dibuat secara tertulis at...
7,5,AYAT,(3) Perjanjian Kerja yang dibuat secara tertulis
8,5,AYAT,(4) Perjanjian Kerja dibuat untuk waktu terten...
9,5,PASAL,Pasal 3


## 8. Structure-aware parser

State yang dipertahankan saat membaca dokumen:
- `document_part`
- `BAB`
- `Bagian`
- `Paragraf`
- `Pasal`
- `Ayat`

`BAB/Bagian/Paragraf` menjadi metadata.

Boundary chunk utama adalah **Pasal/Ayat**.

In [10]:
@dataclass
class PageLine:
    page: int
    text: str


@dataclass
class ParseState:
    document_part: str = "batang_tubuh"
    explanation_section: Optional[str] = None

    chapter: Optional[str] = None
    chapter_title: Optional[str] = None

    subsection: Optional[str] = None
    subsection_title: Optional[str] = None

    paragraph: Optional[str] = None
    paragraph_title: Optional[str] = None

    article_number: Optional[str] = None
    ayat_number: Optional[str] = None

    pending_heading_field: Optional[str] = None
    pending_title_lines: list[str] = field(default_factory=list)


def _flatten_pages(pages: list[list[str]]) -> list[PageLine]:
    return [
        PageLine(page=page_no, text=line)
        for page_no, page_lines in enumerate(pages, start=1)
        for line in page_lines
    ]


def _looks_like_good_split_boundary(line: str) -> bool:
    stripped = line.strip()

    return (
        stripped.endswith((".", ";", ":", "?", "!"))
        or bool(re.match(r"^[a-z]\.", stripped, re.I))
        or bool(re.match(r"^\d+\.", stripped))
    )


def split_unit_lines(lines: list[PageLine], max_chars: int) -> list[list[PageLine]]:
    if not lines:
        return []

    total_chars = sum(len(x.text) + 1 for x in lines)

    if total_chars <= max_chars:
        return [lines]

    chunks = []
    start = 0

    while start < len(lines):
        char_count = 0
        end = start
        best_cut = None

        while end < len(lines):
            next_len = len(lines[end].text) + 1

            if char_count + next_len > max_chars and end > start:
                break

            char_count += next_len
            end += 1

            if _looks_like_good_split_boundary(lines[end - 1].text):
                best_cut = end

        if end >= len(lines):
            chunks.append(lines[start:])
            break

        cut = end

        if best_cut is not None and best_cut > start:
            candidate_chars = sum(len(x.text) + 1 for x in lines[start:best_cut])

            if candidate_chars >= max_chars * 0.55:
                cut = best_cut

        if cut <= start:
            cut = start + 1

        chunks.append(lines[start:cut])
        start = cut

    return chunks


def _state_snapshot(state: ParseState) -> dict:
    return {
        "document_part": state.document_part,
        "explanation_section": state.explanation_section,
        "chapter": state.chapter,
        "chapter_title": state.chapter_title,
        "subsection": state.subsection,
        "subsection_title": state.subsection_title,
        "paragraph": state.paragraph,
        "paragraph_title": state.paragraph_title,
        "article_number": state.article_number,
        "ayat_number": state.ayat_number,
    }


def _section_string(meta: dict) -> Optional[str]:
    parts = []

    if meta.get("explanation_section"):
        parts.append(meta["explanation_section"])

    for label_key, title_key in (
        ("chapter", "chapter_title"),
        ("subsection", "subsection_title"),
        ("paragraph", "paragraph_title"),
    ):
        label = meta.get(label_key)
        title = meta.get(title_key)

        if label:
            parts.append(" ".join(x for x in (label, title) if x))

    return " > ".join(parts) if parts else None


def _embedding_prefix(document_name: str, meta: dict) -> str:
    parts = [document_name]

    if meta["document_part"] == "penjelasan":
        parts.append("Penjelasan")

        if meta.get("explanation_section"):
            parts.append(meta["explanation_section"])

    section = _section_string(meta)

    if section:
        parts.append(section)

    if meta.get("article_number"):
        parts.append(f"Pasal {meta['article_number']}")

    if meta.get("ayat_number"):
        parts.append(f"Ayat ({meta['ayat_number']})")

    return " | ".join(parts)

## 9. Build chunks

In [11]:
def parse_structure_aware_chunks(
    pages: list[list[str]],
    *,
    document_id: str,
    document_name: str,
    max_chars: int = 2500,
) -> list[dict]:

    lines = _flatten_pages(pages)
    state = ParseState()

    current = []
    raw_units = []

    def commit_pending_title():
        if not state.pending_heading_field:
            return

        title = " ".join(state.pending_title_lines).strip() or None
        setattr(state, state.pending_heading_field, title)

        state.pending_heading_field = None
        state.pending_title_lines = []

    def flush_current():
        nonlocal current

        if current:
            raw_units.append((_state_snapshot(state), current.copy()))
            current = []

    for item in lines:
        line = item.text

        # -------------------------
        # PENJELASAN
        # -------------------------
        if line.upper() == "PENJELASAN":
            commit_pending_title()
            flush_current()
            state = ParseState(document_part="penjelasan")
            continue

        if state.document_part == "penjelasan":
            m = EXPL_SECTION_RE.match(line)

            if m:
                commit_pending_title()
                flush_current()

                state.explanation_section = (
                    f"{m.group(1).upper()}. {m.group(2).upper()}"
                )
                state.article_number = None
                state.ayat_number = None
                continue

        # -------------------------
        # BAB
        # -------------------------
        m = BAB_RE.match(line)

        if m and state.document_part == "batang_tubuh":
            commit_pending_title()
            flush_current()

            marker = f"BAB {m.group(1).upper()}"

            if state.chapter != marker:
                state.chapter = marker
                state.chapter_title = None

                state.subsection = None
                state.subsection_title = None

                state.paragraph = None
                state.paragraph_title = None

                state.article_number = None
                state.ayat_number = None

            state.pending_heading_field = "chapter_title"
            state.pending_title_lines = []
            continue

        # -------------------------
        # BAGIAN
        # -------------------------
        m = BAGIAN_RE.match(line)

        if m and state.document_part == "batang_tubuh":
            commit_pending_title()
            flush_current()

            state.subsection = f"Bagian {m.group(1)}"
            state.subsection_title = None

            state.paragraph = None
            state.paragraph_title = None

            state.article_number = None
            state.ayat_number = None

            state.pending_heading_field = "subsection_title"
            state.pending_title_lines = []
            continue

        # -------------------------
        # PARAGRAF
        # -------------------------
        m = PARAGRAF_RE.match(line)

        if m and state.document_part == "batang_tubuh":
            commit_pending_title()
            flush_current()

            state.paragraph = f"Paragraf {m.group(1)}"
            state.paragraph_title = None

            state.article_number = None
            state.ayat_number = None

            state.pending_heading_field = "paragraph_title"
            state.pending_title_lines = []
            continue

        # -------------------------
        # PASAL = hard boundary
        # -------------------------
        m = PASAL_RE.match(line)

        if m:
            commit_pending_title()
            flush_current()

            state.article_number = m.group(1)
            state.ayat_number = None
            continue

        # -------------------------
        # Ayat di PENJELASAN
        # -------------------------
        if state.document_part == "penjelasan" and state.article_number:
            explanation_line = line

            if explanation_line.lower().startswith("ayat (") and ")" not in explanation_line[:12]:
                m_bad = re.match(
                    r"^(Ayat)\s+\((\d{1,2})[1Il](.*)$",
                    explanation_line,
                    re.I,
                )

                if (
                    m_bad
                    and m_bad.group(3)
                    and not m_bad.group(3).startswith(")")
                ):
                    explanation_line = (
                        f"Ayat ({m_bad.group(2)}) "
                        f"{m_bad.group(3).lstrip()}"
                    )

            m = EXPL_AYAT_RE.match(explanation_line)

            if m:
                commit_pending_title()
                flush_current()

                state.ayat_number = m.group(1)
                remainder = m.group(2).strip()

                if remainder:
                    current.append(PageLine(page=item.page, text=remainder))

                continue

        # -------------------------
        # Ayat di BATANG TUBUH
        # -------------------------
        if state.document_part == "batang_tubuh" and state.article_number:
            m = AYAT_RE.match(line)

            if m:
                commit_pending_title()
                flush_current()

                state.ayat_number = m.group(1)
                remainder = m.group(2).strip()

                if remainder:
                    current.append(PageLine(page=item.page, text=remainder))

                continue

        # Title setelah BAB / Bagian / Paragraf
        if state.pending_heading_field:
            state.pending_title_lines.append(line)
            continue

        current.append(item)

    commit_pending_title()
    flush_current()

    # -------------------------
    # Build final chunks
    # -------------------------
    chunks = []

    for meta, unit_lines in raw_units:
        split_groups = split_unit_lines(unit_lines, max_chars=max_chars)

        for split_index, split_lines in enumerate(split_groups, start=1):
            text = "\n".join(x.text for x in split_lines).strip()

            if not text:
                continue

            pages_in_chunk = sorted({x.page for x in split_lines})
            page_start = min(pages_in_chunk)
            page_end = max(pages_in_chunk)

            article_number = meta.get("article_number")
            ayat_number = meta.get("ayat_number")

            part_code = (
                "expl"
                if meta["document_part"] == "penjelasan"
                else "body"
            )

            id_parts = [document_id, part_code]

            id_parts.append(
                f"pasal{article_number.lower()}"
                if article_number
                else "section"
            )

            if ayat_number:
                id_parts.append(f"ayat{ayat_number}")

            id_parts.append(f"{split_index:02d}")

            prefix = _embedding_prefix(document_name, meta)

            short_hash = hashlib.sha1(
                f"{prefix}\n{text}".encode("utf-8")
            ).hexdigest()[:8]

            chunk_id = "-".join(id_parts) + f"-{short_hash}"

            chunks.append(
                {
                    # Metadata minimum PRD
                    "document_id": document_id,
                    "document_name": document_name,
                    "page_number": page_start,
                    "section": _section_string(meta),
                    "article_number": article_number,
                    "chunk_id": chunk_id,
                    "text": text,

                    # Extra metadata
                    "document_part": meta["document_part"],
                    "page_start": page_start,
                    "page_end": page_end,
                    "pages": pages_in_chunk,

                    "chapter": meta.get("chapter"),
                    "chapter_title": meta.get("chapter_title"),

                    "subsection": meta.get("subsection"),
                    "subsection_title": meta.get("subsection_title"),

                    "paragraph": meta.get("paragraph"),
                    "paragraph_title": meta.get("paragraph_title"),

                    "article": (
                        f"Pasal {article_number}"
                        if article_number
                        else None
                    ),

                    "ayat_number": ayat_number,
                    "ayat": (
                        f"Ayat ({ayat_number})"
                        if ayat_number
                        else None
                    ),

                    "chunk_part": split_index,

                    # Field yang direkomendasikan untuk embedding
                    "embedding_text": f"{prefix}\n{text}",
                }
            )

    return chunks

## 10. Jalankan parser ke kedua PDF

In [12]:
def parse_document(
    source: SourceDocument,
    *,
    max_chars: int = 2500,
    enable_ocr: bool = True,
    ocr_lang: str = "ind+eng",
) -> list[dict]:

    pages = extract_pdf_pages(
        source.path,
        enable_ocr=enable_ocr,
        ocr_lang=ocr_lang,
    )

    return parse_structure_aware_chunks(
        pages,
        document_id=source.document_id,
        document_name=source.document_name,
        max_chars=max_chars,
    )


all_chunks = []

for source in SOURCES:
    chunks = parse_document(source)
    all_chunks.extend(chunks)

    print(
        source.document_id,
        "| chunks:", len(chunks),
        "| ayat chunks:", sum(x["ayat_number"] is not None for x in chunks),
        "| penjelasan:", sum(x["document_part"] == "penjelasan" for x in chunks),
    )

print("\nTOTAL CHUNKS:", len(all_chunks))

uu13_2003 | chunks: 843 | ayat chunks: 632 | penjelasan: 344


pp35_2021 | chunks: 258 | ayat chunks: 184 | penjelasan: 95

TOTAL CHUNKS: 1101


## 11. Inspect dataset hasil chunking

In [13]:
chunks_df = pd.DataFrame(all_chunks)

display(
    chunks_df[
        [
            "document_id",
            "document_part",
            "page_start",
            "page_end",
            "chapter",
            "subsection",
            "article",
            "ayat",
            "chunk_part",
            "chunk_id",
        ]
    ].head(30)
)

,document_id,document_part,page_start,page_end,chapter,subsection,article,ayat,chunk_part,chunk_id
0,uu13_2003,batang_tubuh,1,2,None,None,None,None,1,uu13_2003-body-section-01-95e76aff
1,uu13_2003,batang_tubuh,2,3,BAB I,None,Pasal 1,None,1,uu13_2003-body-pasal1-01-5dc81be4
2,uu13_2003,batang_tubuh,3,5,BAB I,None,Pasal 1,None,2,uu13_2003-body-pasal1-02-2b1c9256
3,uu13_2003,batang_tubuh,5,6,BAB I,None,Pasal 1,None,3,uu13_2003-body-pasal1-03-e07f99f0
4,uu13_2003,batang_tubuh,6,6,BAB II,None,Pasal 2,None,1,uu13_2003-body-pasal2-01-6afa56bb
5,uu13_2003,batang_tubuh,6,6,BAB II,None,Pasal 3,None,1,uu13_2003-body-pasal3-01-70ebbc2d
6,uu13_2003,batang_tubuh,6,6,BAB II,None,Pasal 4,None,1,uu13_2003-body-pasal4-01-168e6755
7,uu13_2003,batang_tubuh,7,7,BAB III,None,Pasal 5,None,1,uu13_2003-body-pasal5-01-c3e3fdf3
8,uu13_2003,batang_tubuh,7,7,BAB III,None,Pasal 6,None,1,uu13_2003-body-pasal6-01-baba28ed
9,uu13_2003,batang_tubuh,7,7,BAB IV,None,Pasal 7,Ayat (1),1,uu13_2003-body-pasal7-ayat1-01-e4777dd8


### Distribusi chunk

In [14]:
summary = (
    chunks_df
    .groupby(["document_id", "document_part"], dropna=False)
    .agg(
        chunks=("chunk_id", "count"),
        ayat_chunks=("ayat_number", lambda s: s.notna().sum()),
        min_page=("page_start", "min"),
        max_page=("page_end", "max"),
    )
    .reset_index()
)

display(summary)

,document_id,document_part,chunks,ayat_chunks,min_page,max_page
0,pp35_2021,batang_tubuh,163,139,1,42
1,pp35_2021,penjelasan,95,45,43,56
2,uu13_2003,batang_tubuh,499,428,1,79
3,uu13_2003,penjelasan,344,204,80,128


## 12. Contoh konkret: PP 35/2021 — Pasal 8

Bagian ini penting untuk sanity check.

Kita harapkan Ayat (1), Ayat (2), Ayat (3) menjadi chunk terpisah dan citation page tetap terjaga.

In [15]:
pasal8 = chunks_df[
    (chunks_df["document_id"] == "pp35_2021")
    & (chunks_df["document_part"] == "batang_tubuh")
    & (chunks_df["article_number"] == "8")
][
    [
        "article",
        "ayat",
        "page_start",
        "page_end",
        "section",
        "text",
        "embedding_text",
        "chunk_id",
    ]
]

display(pasal8)

,article,ayat,page_start,page_end,section,text,embedding_text,chunk_id
860,Pasal 8,Ayat (1),7,7,BAB II PERJANJIAN KERJA WAKTU TERTENTU > Bagia...,PKWT berdasarkan jangka waktu sebagaimana\ndim...,"PP No. 35 Tahun 2021 tentang PKWT, Alih Daya, ...",pp35_2021-body-pasal8-ayat1-01-35f07828
861,Pasal 8,Ayat (2),7,7,BAB II PERJANJIAN KERJA WAKTU TERTENTU > Bagia...,Dalam hal jangka waktu PKWT sebagaimana dimaks...,"PP No. 35 Tahun 2021 tentang PKWT, Alih Daya, ...",pp35_2021-body-pasal8-ayat2-01-942021f6
862,Pasal 8,Ayat (3),8,8,BAB II PERJANJIAN KERJA WAKTU TERTENTU > Bagia...,Masa kerja Pekerja/Buruh dalam hal perpanjanga...,"PP No. 35 Tahun 2021 tentang PKWT, Alih Daya, ...",pp35_2021-body-pasal8-ayat3-01-d86e6eeb


### Print satu chunk secara penuh

In [16]:
example = pasal8.iloc[1].to_dict()

print("CHUNK ID:")
print(example["chunk_id"])

print("\nTEXT:")
print(example["text"])

print("\nEMBEDDING TEXT:")
print(example["embedding_text"])

CHUNK ID:
pp35_2021-body-pasal8-ayat2-01-942021f6

TEXT:
Dalam hal jangka waktu PKWT sebagaimana dimaksud
pada ayat (1) akan berakhir dan pekerjaan yang
dilaksanakan belum selesai maka dapat dilakukan
perpanjangan PKWT dengan jangka waktu sesuai
kesepakatan antara Pengusaha dengan
Pekerja/Buruh, dengan ketentuan jangka waktu
keseluruhan PKWT beserta perpanjangannya tidak
lebih dari 5 (lima) tahun.

EMBEDDING TEXT:
PP No. 35 Tahun 2021 tentang PKWT, Alih Daya, Waktu Kerja, dan PHK | BAB II PERJANJIAN KERJA WAKTU TERTENTU > Bagian Kedua Pelaksanaan Perjanjian Kerja Waktu Tertentu | Pasal 8 | Ayat (2)
Dalam hal jangka waktu PKWT sebagaimana dimaksud
pada ayat (1) akan berakhir dan pekerjaan yang
dilaksanakan belum selesai maka dapat dilakukan
perpanjangan PKWT dengan jangka waktu sesuai
kesepakatan antara Pengusaha dengan
Pekerja/Buruh, dengan ketentuan jangka waktu
keseluruhan PKWT beserta perpanjangannya tidak
lebih dari 5 (lima) tahun.


## 13. Validation sebelum disimpan

Kita cek invariant minimum:
- `chunk_id` unik
- text tidak kosong
- ada document metadata
- page number valid
- chunk Pasal/Ayat memiliki article metadata

In [17]:
assert len(all_chunks) > 0
assert chunks_df["chunk_id"].is_unique
assert chunks_df["text"].fillna("").str.strip().ne("").all()
assert chunks_df["document_id"].notna().all()
assert chunks_df["document_name"].notna().all()
assert chunks_df["page_number"].ge(1).all()
assert chunks_df["embedding_text"].fillna("").str.strip().ne("").all()

print("✅ Validation passed")
print("Unique chunk IDs :", chunks_df["chunk_id"].nunique())
print("Total chunks     :", len(chunks_df))
print("Empty texts      :", chunks_df["text"].fillna("").str.strip().eq("").sum())

✅ Validation passed
Unique chunk IDs : 1101
Total chunks     : 1101
Empty texts      : 0


## 14. Save dataset `.jsonl`

Satu baris = satu JSON object/chunk.

Format JSONL enak untuk:
- streaming ingestion,
- debugging,
- batch embedding,
- indexing ke Qdrant.

In [18]:
def write_jsonl(chunks: Iterable[dict], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with output_path.open("w", encoding="utf-8") as f:
        for chunk in chunks:
            f.write(json.dumps(chunk, ensure_ascii=False) + "\n")


write_jsonl(all_chunks, OUTPUT_PATH)

print("Saved:", OUTPUT_PATH)
print("Size :", f"{OUTPUT_PATH.stat().st_size / 1024:.1f} KB")

Saved: /mnt/data/labor_law_chunks_notebook.jsonl
Size : 1234.5 KB


## 15. Preview isi JSONL

In [19]:
with OUTPUT_PATH.open("r", encoding="utf-8") as f:
    for i in range(3):
        obj = json.loads(next(f))
        print(json.dumps(obj, ensure_ascii=False, indent=2))
        print("-" * 100)

{
  "document_id": "uu13_2003",
  "document_name": "UU No. 13 Tahun 2003 tentang Ketenagakerjaan",
  "page_number": 1,
  "section": null,
  "article_number": null,
  "chunk_id": "uu13_2003-body-section-01-95e76aff",
  "text": "UNDANG-UNDANG REPUBLIK INDONESIA\nNOMOR 13 TAHUN 2003\nTENTANG\nKETENAGAKERJAAN\nDENGAN RAHMAT TUHAN YANG MAHA ESA\nPRESIDEN REPUBLIK INDONESIA,\nMenimbang : a. bahwa pembangunan nasional dilaksanakan dalam rangka\npembangunan manusia Indonesia seutuhnya dan pembangunan\nmasyarakat Indonesia seluruhnya untuk mewujudkan masyarakat yang\nsejahtera, adil, makmur, yang merata, baik materiil maupun spiritual\nberdasarkan Pancasila dan Undang Undang Dasar Negara Republik\nIndonesia Tahun 1945;\nb. bahwa dalam pelaksanaan pembangunan nasional, tenaga kerja\nmempunyai peranan dan kedudukan yang sangat penting sebagai\npelaku dan tujuan pembangunan;\nc. bahwa sesuai dengan peranan dan kedudukan tenaga kerja, diperlukan\npembangunan ketenagakerjaan untuk meningkatkan kuali

## 16. Output untuk tahap embedding

Field yang sebaiknya dipakai sebagai input **BGE-M3**:

```python
chunk["embedding_text"]
```

Sedangkan:

```python
chunk["text"]
```

tetap disimpan sebagai source context yang bersih untuk dikirim ke reranker/LLM.

Tahap selanjutnya:

```text
labor_law_chunks_notebook.jsonl
→ BGE-M3 dense + sparse
→ Qdrant hybrid index
```